In [ ]:
import pandas as pd
from pathlib import Path

# ─── Configuración ────────────────────────────────────────────────────────────
MANIFEST_PATH = r"C:\Users\trodr\Documents\proyecto-torax\01-dataset\archive\official_data_iccv_final\SHA256SUMS.txt"
BASE_DIR      = r"C:\Users\trodr\Documents\proyecto-torax\01-dataset\archive\official_data_iccv_final"

# ─── Parsear el manifest ───────────────────────────────────────────────────────
def parse_manifest(manifest_path, base_dir):
    rows = []
    with open(manifest_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # Separar hash y ruta (separados por espacio o tab)
            parts = line.split(None, 1)  # split en primer whitespace
            if len(parts) != 2:
                continue
            
            sha256, rel_path = parts
            
            # Solo quedarse con imágenes .jpg
            if not rel_path.lower().endswith('.jpg'):
                continue
            
            # Construir full_path
            full_path = str(Path(base_dir) / rel_path.replace('/', '\\'))
            
            # Extraer subject_id y study_id del path
            # Estructura: files/p10/p10000032/s50414267/imagen.jpg
            path_parts = Path(rel_path).parts
            # path_parts = ('files', 'p10', 'p10000032', 's50414267', 'imagen.jpg')
            
            subject_id = int(path_parts[2].lstrip('p'))  # p10000032 → 10000032
            study_id   = int(path_parts[3].lstrip('s'))  # s50414267 → 50414267
            image_id   = Path(rel_path).stem              # UUID sin .jpg
            
            rows.append({
                'sha256':     sha256,
                'subject_id': subject_id,
                'study_id':   study_id,
                'image_id':   image_id,
                'rel_path':   rel_path,
                'full_path':  full_path,
            })
    
    return pd.DataFrame(rows)


# ─── Cargar y mergear con tu CSV de labels ────────────────────────────────────
df_paths  = parse_manifest(MANIFEST_PATH, BASE_DIR)
print(f"Imágenes encontradas en manifest: {len(df_paths)}")

# Tu CSV de labels (ajusta la ruta)
LABELS_CSV = r"C:\Users\trodr\Documents\proyecto-torax\01-dataset\archive\official_data_iccv_final\chexpert.csv"
df_labels  = pd.read_csv(LABELS_CSV)

# Merge por subject_id + study_id
df = df_paths.merge(df_labels, on=['subject_id', 'study_id'], how='inner')
print(f"Filas tras merge con labels: {len(df)}")
print(df[['subject_id', 'study_id', 'full_path'] + LABEL_COLS].head())

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

base_dir = r"C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset_prueba\subset_pa_ap"

# Primero contar total para la barra
subjects = [s for s in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, s))]

image_paths = []

for subject_id in tqdm(subjects, desc="Sujetos"):
    subject_path = os.path.join(base_dir, subject_id)
    
    studies = [s for s in os.listdir(subject_path) if os.path.isdir(os.path.join(subject_path, s))]
    
    for study_id in tqdm(studies, desc=f"  {subject_id}", leave=False):
        study_path = os.path.join(subject_path, study_id)
        
        for fname in os.listdir(study_path):
            if fname.endswith(".jpg"):
                full_path = os.path.join(study_path, fname)
                image_paths.append({
                    "subject_id": subject_id,
                    "study_id": study_id,
                    "full_path": full_path
                })

df_paths = pd.DataFrame(image_paths)
print(f"\nTotal imágenes encontradas: {len(df_paths)}")
print(df_paths.head())

Sujetos: 100%|██████████| 285/285 [00:00<00:00, 298.33it/s]


Total imágenes encontradas: 0
Empty DataFrame
Columns: []
Index: []


In [4]:
image_paths = []

for subject_id in tqdm(subjects, desc="Sujetos"):
    subject_path = os.path.join(base_dir, subject_id)
    
    for fname in os.listdir(subject_path):
        if fname.endswith(".jpg"):
            full_path = os.path.join(subject_path, fname)
            image_paths.append({
                "subject_id": subject_id,
                "full_path": full_path
            })

df_paths = pd.DataFrame(image_paths)
print(f"Total imágenes encontradas: {len(df_paths)}")
print(df_paths.head())

Sujetos: 100%|██████████| 285/285 [00:00<00:00, 16274.03it/s]

Total imágenes encontradas: 315
  subject_id                                          full_path
0  s50041967  C:\Users\trodr\Documents\proyecto-torax-v2.0\0...
1  s50100841  C:\Users\trodr\Documents\proyecto-torax-v2.0\0...
2  s50125601  C:\Users\trodr\Documents\proyecto-torax-v2.0\0...
3  s50130130  C:\Users\trodr\Documents\proyecto-torax-v2.0\0...
4  s50240426  C:\Users\trodr\Documents\proyecto-torax-v2.0\0...
